### Google Next 26 DA 영역 신기능 톺아보기

#### 개요
*   본 노트북은 한국 리전의 GCP 유저들이 DA 영역에 대한 신기능을 빠르고 쉽게 이해할 수 있도록 제작 되었습니다.
*   Infra 측면의 기능은 별도로 참고하십시오. 해당 노트북은 BigQuery에서 User가 실제로 작업하는 과정만 다룹니다.
*   Lakehouse(구. BigLake) 및 Lightning Engine for Apache Spark 관련은 [LINK](https://github.com/changsub214/icebergingcp) 참고하세요.

#### 주의
*   최대한의 과정은 Run만 누르면 동작하도록 제작되었으나 일부 과정은 직접 생성 및 수정이 필요할 수 있습니다.

#### 목차


1.   [BigQuery][GA] AI Functions
- [BigQuery][GA] TimesFM => AI Functions의 AI.FORECAST 함수로 적용
2.   [BigQuery][Preview] Zero-Shot Tabular FM    
3.   [BigQuery][GA] ObjectRef   
4.   [BigQuery][GA] Python UDFs
5.   [BigQuery][Preview] Knowledge Graph
6.   [BigQuery][Preview] Hybrid Search
7.   [BigQuery][GA] Conversational Analytics in BigQuery



#### 환경설정

*   실습에 필요한 환경변수 설정합니다.
*   실습에 필요한 데이터 세트를 생성합니다.

In [20]:
### 실습위한 환경변수 세팅
import google.auth
import subprocess
import requests
import json
import sys
from google.cloud import bigquery

### Project의 ID 및 account 정보 획득
project_id = google.auth.default()[1]
user_account = !gcloud config get-value account
user_account = user_account[0]

### 변수 설정
region = "us-central1"
data_region = "us"

### 검증
print("🏁 다음과 같이 할당 완료되었습니다.")
print(f"✅ project_id : {project_id}")
#print(f"✅ user_account : {user_account}")
print(f"✅ region : {region}")

🏁 다음과 같이 할당 완료되었습니다.
✅ project_id : krda-next26
✅ region : us-central1


In [ ]:
### BigQuery 연결
print(f"🚀 BigQuery 연결 및 Data 생성을 시작합니다.")
client = bigquery.Client(project=project_id)

### BigQuery 데이터 생성
### 데이터는 Google Public Data인 The Look e-commerce 데이터를 활용합니다.
dataset_id = f"{project_id}.ecommerce"
dataset = bigquery.Dataset(dataset_id)
dataset.location = "US"  # Public Data가 US에 있으므로 US로 설정

try:
    client.create_dataset(dataset, timeout=30)
    print(f"✅ 데이터세트 {dataset_id}가 생성 완료되었습니다.")
except Exception as e:
    print(f"⚠️ 데이터세트 관련 알림: {e}")

table_names = [
    "inventory_items",
    "events",
    "users",
    "products",
    "orders",
    "order_items",
    "distribution_centers"
]

print(f"📊 총 {len(table_names)}개의 테이블 복사를 시작합니다.")

for table_name in table_names:
    destination_table = f"{dataset_id}.{table_name}"

    job_config = bigquery.QueryJobConfig(
        destination=destination_table,
        write_disposition="WRITE_TRUNCATE"
    )

    sql = f"""
        SELECT *
        FROM `bigquery-public-data.thelook_ecommerce.{table_name}`
    """

    try:
        query_job = client.query(sql, job_config=job_config)
        query_job.result()
        print(f"  └ ✅ 테이블 생성 성공: {destination_table}")
    except Exception as e:
        print(f"  └ ❌ 테이블 생성 실패 ({table_name}): {e}")

print("\n✨ 데이터 생성 작업이 완료되었습니다!")

🚀 BigQuery 연결 및 Data 생성을 시작합니다.
⚠️ 데이터세트 관련 알림: 409 POST https://bigquery.googleapis.com/bigquery/v2/projects/krda-next26/datasets?prettyPrint=false: Already Exists: Dataset krda-next26:ecommerce
📊 총 7개의 테이블 복사를 시작합니다.
  └ ✅ 테이블 생성 성공: krda-next26.ecommerce.inventory_items
  └ ✅ 테이블 생성 성공: krda-next26.ecommerce.events
  └ ✅ 테이블 생성 성공: krda-next26.ecommerce.users
  └ ✅ 테이블 생성 성공: krda-next26.ecommerce.products
  └ ✅ 테이블 생성 성공: krda-next26.ecommerce.orders
  └ ✅ 테이블 생성 성공: krda-next26.ecommerce.order_items
  └ ✅ 테이블 생성 성공: krda-next26.ecommerce.distribution_centers

✨ 데이터 생성 작업이 완료되었습니다!


#### BigQuery Features

##### **AI Funtions**

\# AI \# BQML \# SQL

What is AI Functions?  
BigQuery에서 복잡한 ML 파이프라인 구성 없이 손쉽게 SQL 구문을 통해 AI를 활용할 수 있는 기능입니다.  
*   [GA][AI.GENERATE](https://docs.cloud.google.com/bigquery/docs/reference/standard-sql/bigqueryml-syntax-ai-generate) : 데이터를 생성합니다.
*   [GA][AI.CLASSIFY](https://docs.cloud.google.com/bigquery/docs/reference/standard-sql/bigqueryml-syntax-ai-classify) : 제공된 카테고리를 바탕으로 분류되지 않은 데이터를 분류합니다.
*   [GA][Autonomous Embedding Generation](https://docs.cloud.google.com/bigquery/docs/autonomous-embedding-generation) : 임베딩 값을 자동 생성합니다.
*   [GA][AI.EMBED](https://docs.cloud.google.com/bigquery/docs/reference/standard-sql/bigqueryml-syntax-ai-embed) : 데이터를 벡터 임베딩합니다.
*   [Preview][AI.SEARCH](https://docs.cloud.google.com/bigquery/docs/reference/standard-sql/bigqueryml-syntax-ai-search) : 자율 임베딩 된 데이터를 검색합니다.
*   [GA][AI.IF](https://docs.cloud.google.com/bigquery/docs/reference/standard-sql/bigqueryml-syntax-ai-if) : 자연어를 사용해 데이터의 의미론적 필터링 및 조인을 수행합니다. 복잡한 WHERE, JOIN, LIKE, REGEXP 구문 사용을 피할 수 있습니다.
*   [GA][AI.SCORE](https://docs.cloud.google.com/bigquery/docs/reference/standard-sql/bigqueryml-syntax-ai-score) : 자연어 기준을 바탕으로 행의 점수를 매기고 순위를 지정합니다.
*   [GA][AI.FORECAST](https://docs.cloud.google.com/bigquery/docs/reference/standard-sql/bigqueryml-syntax-ai-forecast) : TimesFM 모델을 사용해 시계열 예측합니다.
*   [Preview]AI.PARSE_DOCUMENT : Document AI를 사용해 문서를 파싱합니다.  
*   [Preview][AI.COUNT_TOKENS](https://docs.cloud.google.com/bigquery/docs/reference/standard-sql/bigqueryml-syntax-ai-count-tokens) : 사전에 토큰 수를 계산해 비용을 추정합니다.  
*   [Preview][AI.DETECT_ANOMALIES](https://docs.cloud.google.com/bigquery/docs/reference/standard-sql/bigqueryml-syntax-ai-detect-anomalies) : TimesFM 기반으로 별도 모델 학습 필요 없이 이상치를 찾아냅니다.  

What is benefits?

*   복잡한 ML 파이프라인 구성없이 손쉽게 SQL 구문으로 데이터 분석 및 증강이 가능합니다.

In [ ]:
%%bigquery
-- AI.GENERATE을 수행할 Table의 스키마 일부를 간략하게 살펴봅니다.
SELECT
  id,
  name,
  category
FROM `ecommerce.products`
LIMIT 5

Query is running:   0%|          |

Downloading:   0%|          |

,id,name,category
0,13844,(ONE) 1 Satin Headband,Accessories
1,14086,(One) CHEER Rhinestone Studded Stretch Headband,Accessories
2,13726,(Set of 3) Leopard Animal Print Stretch Headband,Accessories
3,14106,(Set of 4) 2.5 Glitter Cotton Stretch Headbands,Accessories
4,28405,1 1/2 In. Original Perry Suspenders,Accessories


In [ ]:
%%bigquery
-- AI.GENERATE는 Gemini를 기반으로 프롬프트에 지시된 작업을 손쉽게 수행하는 강력한 함수입니다.
SELECT
  id,
  name,
  category,
  -- AI.GENERATE가 반환하는 STRUCT 객체에서 result 값만 추출
  (AI.GENERATE(
    CONCAT('다음 상품을 판매하기 위한 매력적인 SNS 마케팅 문구를 이모지와 함께 1줄로 작성해줘. 상품명: ', name, ', 카테고리: ', category)
  )).result AS marketing_slogan
FROM `ecommerce.products`
LIMIT 5


Query is running:   0%|          |

Downloading:   0%|          |

,id,name,category,marketing_slogan
0,13726,(Set of 3) Leopard Animal Print Stretch Headband,Accessories,🐆✨ 힙한 레오파드 헤어밴드 3종으로 어떤 룩이든 스타일리시하게 완성!
1,13844,(ONE) 1 Satin Headband,Accessories,✨ 찰랑이는 머릿결과 우아한 스타일! (ONE) 새틴 헤어밴드 하나로 완성하세요 💖
2,14106,(Set of 4) 2.5 Glitter Cotton Stretch Headbands,Accessories,매일매일 반짝이는 스타일! ✨ 편안한 코튼 스트레치 글리터 헤어밴드 4종 세트 💖
3,14086,(One) CHEER Rhinestone Studded Stretch Headband,Accessories,반짝이는 라인스톤과 편안함까지! ✨(One) CHEER✨ 헤어밴드로 당신의 하루를 ...
4,28405,1 1/2 In. Original Perry Suspenders,Accessories,오리지널 페리 서스펜더로 흐트러짐 없는 클래식 룩 완성! 🕺✨


In [ ]:
%%bigquery
-- AI.CLASSIFY를 수행할 Table의 스키마 일부를 간략하게 살펴봅니다.
SELECT
  distinct user_id,
  city
FROM `ecommerce.events`
WHERE user_id IS NOT NULL
ORDER BY user_id
LIMIT 5

Query is running:   0%|          |

Downloading:   0%|          |

,user_id,city
0,1,Kitahiroshima City
1,3,Avilés
2,4,Seoul
3,5,Shanghai
4,7,Tianjin


In [ ]:
%%bigquery
-- AI.CLASSIFY는 데이터를 기반으로 분류를 손쉽게 할 수 있는 함수입니다.
WITH TargetUsers AS (
  SELECT DISTINCT
    user_id,
    city
  FROM `ecommerce.events`
  WHERE user_id IS NOT NULL
  ORDER BY user_id
  LIMIT 5
)

SELECT
  user_id,
  city,
  AI.CLASSIFY(
    CONCAT('도시: ', city),
    ['동북아시아', '동남아시아', '중앙아시아', '서아시아', '북아메리카', '남아메리카', '유럽', '아프리카', '오세아니아']
  ) AS country_category
FROM TargetUsers
ORDER BY user_id

Query is running:   0%|          |

Downloading:   0%|          |

,user_id,city,country_category
0,1,Kitahiroshima City,동북아시아
1,3,Avilés,유럽
2,4,Seoul,동북아시아
3,5,Shanghai,동북아시아
4,7,Tianjin,동북아시아


In [ ]:
%%bigquery
-- AI.EMBED를 수행할 Table을 살펴봅니다.
-- 상품 명과 상품의 브랜드를 나타냅니다.
SELECT
  id,
  name,
  brand
FROM `ecommerce.products`
LIMIT 5

Query is running:   0%|          |

Downloading:   0%|          |

,id,name,brand
0,13844,(ONE) 1 Satin Headband,Funny Girl Designs
1,14086,(One) CHEER Rhinestone Studded Stretch Headband,Funny Girl Designs
2,13726,(Set of 3) Leopard Animal Print Stretch Headband,Funny Girl Designs
3,14106,(Set of 4) 2.5 Glitter Cotton Stretch Headbands,Funny Girl Designs
4,28405,1 1/2 In. Original Perry Suspenders,Perry


In [ ]:
%%bigquery
-- AI.EMBED를 통해 손쉽게 벡터 임베딩 값으로 변환할 수 있습니다.
-- 임베딩을 수행하면 추후 유사 상품 추천 및 의미 기반 검색을 수행할 수 있는 기반이 됩니다.
-- AI.EMBED의 핵심은 별도의 커넥터, 모델 생성없이 모델명을 지정하면 사용할 수 있다는 점입니다.
CREATE OR REPLACE TABLE `ecommerce.embeded_product` (
  id INT64,
  name STRING,
  brand STRING,
  department STRING,
  category STRING,
  -- 하기 컬럼은 BigQuery가 알아서 관리합니다.
  -- AI.SEARCH()은 자율 임베딩이 필요로 합니다.
  -- Apr.29.2026 : Autonomous Embedding Generation은 GA, AI.SEARCH()는 Preview인 기능입니다.
  name_embedding STRUCT<result ARRAY<FLOAT64>, status STRING>
    GENERATED ALWAYS AS (
      AI.EMBED(name, model => 'embeddinggemma-300m')
    ) STORED OPTIONS( asynchronous = TRUE ))

Query is running:   0%|          |

""


In [ ]:
%%bigquery
-- INSERT를 진행할 때 name_embedding은 넣지 않습니다. 자율적으로 임베딩 되는 컬럼이기 때문입니다.
INSERT INTO `ecommerce.embeded_product` (id, name, brand, department, category) VALUES
  (5364678,'Leopard hearband', 'SolarClothes', 'Women', 'Accessories')
INSERT INTO `ecommerce.embeded_product` (id, name, brand, department, category) VALUES
  (5364680,'Cool Red pants', 'Pants', 'Men', 'Pants')

Query is running:   0%|          |

""


In [ ]:
%%bigquery
-- AI.SEARCH는 손쉽게 데이터를 검색할 수 있는 함수이며 자율임베딩된 값에서 동작합니다.
SELECT *
FROM AI.SEARCH(
  TABLE `ecommerce.embeded_product`,
  'name',                         -- 테이블 내 대상 컬럼
  "Find products related to Leopard",
  top_k => 10                   -- 유사한 갯수 지정
)

Query is running:   0%|          |

Downloading:   0%|          |

,base,distance
0,"{'id': 5364678, 'name': 'Leopard hearband', 'b...",0.974066


In [ ]:
%%bigquery
-- AI.IF를 수행할 테이블을 간략하게 살펴봅니다.
SELECT
  id,
  name,
  department,
  category
FROM `ecommerce.products`
LIMIT 5

Query is running:   0%|          |

Downloading:   0%|          |

,id,name,department,category
0,13844,(ONE) 1 Satin Headband,Women,Accessories
1,14086,(One) CHEER Rhinestone Studded Stretch Headband,Women,Accessories
2,13726,(Set of 3) Leopard Animal Print Stretch Headband,Women,Accessories
3,14106,(Set of 4) 2.5 Glitter Cotton Stretch Headbands,Women,Accessories
4,28405,1 1/2 In. Original Perry Suspenders,Men,Accessories


In [ ]:
%%bigquery
-- 위 테이블에서 Women 및 Accessories인 항목 5개를 오름차순으로 필터링 합니다.
-- 해당 쿼리는 아래 AI.IF 필터링이 잘 동작하는지 검증하기 위한 용도입니다.
SELECT
  id,
  name,
  department,
  category
FROM `ecommerce.products`
WHERE department = 'Women' and category = 'Accessories'
ORDER BY id
LIMIT 5

Query is running:   0%|          |

Downloading:   0%|          |

,id,name,department,category
0,13595,Scarfand's Leopard Infinity Scarf,Women,Accessories
1,13596,Ray-Ban RB2132 New Wayfarer Sunglasses,Women,Accessories
2,13597,Ray Ban RB3025 Aviator Sunglasses,Women,Accessories
3,13598,Super Soft Cashmere Feel Classic Plaid Tassel ...,Women,Accessories
4,13599,Premium Pashmina Shawl Wrap Scarf by Tapp Coll...,Women,Accessories


In [ ]:
%%bigquery
-- Ai.IF를 통해서 복잡한 LIKE, REGEXP 구문을 쓰지 않고 자연어를 사용해 필터링하여 조회합니다.
-- AI.IF를 잘 사용하려면 SQL를 통해 단순 일치 조건 필터링을 수행하고 의미론적 필터링으로 활용하는 것이 좋습니다. (성능/비용최적화)
WITH WomenAcc AS (
  SELECT
    id,
    name,
    department,
    category
  FROM `ecommerce.products`
  WHERE department = 'Women' and category = 'Accessories'
  ORDER BY id
  LIMIT 5
)

SELECT
  id,
  name,
  department,
  category
FROM WomenAcc
WHERE AI.IF(
  CONCAT("여성이 하는 악세사리중에 선글라스 필터링 해줘 상품명은", name)
)

Query is running:   0%|          |

Downloading:   0%|          |

,id,name,department,category
0,13597,Ray Ban RB3025 Aviator Sunglasses,Women,Accessories
1,13596,Ray-Ban RB2132 New Wayfarer Sunglasses,Women,Accessories


In [ ]:
%%bigquery
--AI.IF를 JOIN ... ON 절에 사용하면 조인의 키 값이 일치하지 않아도 텍스트의 문맥상 서로 연관된 데이터끼리 결합할 수 있는 강력한 기능으로 활용할 수 있습니다.
--고객이 문의하고있는 상품의 후보군을 찾는 예시입니다.
WITH customer_inquiries AS (
  SELECT
    1 AS ticket_id,
    'Men' AS department,
    'Calvin Klein' AS brand,
    '가슴 쪽에 로고가 작게 박힌 검은색 반팔 티셔츠 샀는데 사이즈 교환 되나요?' AS message
  UNION ALL
  SELECT
    2 AS ticket_id,
    'Women' AS department,
    'Columbia' AS brand,
    '작년에 산 핑크색 겨울용 플리스 자켓 지퍼가 고장났어요.' AS message
)

SELECT
  i.ticket_id,
  i.message AS customer_message,
  p.id AS matched_product_id,
  p.name AS matched_product_name,
  p.category
FROM customer_inquiries AS i
JOIN `ecommerce.products` AS p
  ON i.brand = p.brand
  AND i.department = p.department
  AND AI.IF(
    CONCAT(
      "고객의 문의 내용: '", i.message, "'. ",
      "이 고객이 문의하고 있는 상품으로 보이는 후보군이 맞습니까? ",
      "상품명: ", p.name, ", 카테고리: ", p.category
    )
  )
LIMIT 5


Query is running:   0%|          |

Downloading:   0%|          |

,ticket_id,customer_message,matched_product_id,matched_product_name,category
0,2,작년에 산 핑크색 겨울용 플리스 자켓 지퍼가 고장났어요.,8284,Columbia Women's Benton Springs Full Zip Jacke...,Outerwear & Coats
1,2,작년에 산 핑크색 겨울용 플리스 자켓 지퍼가 고장났어요.,8374,Columbia Women's Glacial Fleece III Full Zip J...,Outerwear & Coats
2,1,가슴 쪽에 로고가 작게 박힌 검은색 반팔 티셔츠 샀는데 사이즈 교환 되나요?,16433,Calvin Klein Sportswear Men's Short Sleeve Cre...,Tops & Tees
3,1,가슴 쪽에 로고가 작게 박힌 검은색 반팔 티셔츠 샀는데 사이즈 교환 되나요?,25550,Calvin Klein Mens Body 3 Pack Slim Fit Short S...,Underwear
4,2,작년에 산 핑크색 겨울용 플리스 자켓 지퍼가 고장났어요.,8540,Columbia Women's Fast Trek Ii Full Zip Fleece ...,Outerwear & Coats


In [ ]:
%%bigquery
-- AI.SCORE를 통해서 자연어로 손쉽게 점수를 산정해 Ranking 등에 활용할 수 있습니다.
WITH FilteredTable AS (
  SELECT *
  FROM `ecommerce.order_items`
  WHERE status = 'Returned'
  LIMIT 50
)

SELECT
  order_id,
  product_id,
  status,
  sale_price,
  AI.SCORE((
    """
    반품된 상품의 가격이 비쌀수록 비즈니스 매출 손실이 큽니다. 타격이 전혀 없으면 1점 치명적일 수록 높게 점수를 매기세요. 가격:
    """, CAST(sale_price AS STRING))
  ) AS business_impact_score
FROM FilteredTable
LIMIT 5

Query is running:   0%|          |

Downloading:   0%|          |

,order_id,product_id,status,sale_price,business_impact_score
0,50605,12659,Returned,3.28,4.0
1,58553,15395,Returned,2.67,3.0
2,74305,12265,Returned,2.99,3.0
3,13979,11027,Returned,3.80,5.0
4,82115,13662,Returned,2.95,3.0


In [ ]:
%%bigquery
-- AI.FORECAST는 TimesFM을 기반하여 시계열 예측하는 함수입니다.
-- 주문일자별 주문량을 바탕으로 향후 주문량을 에측하는 예시입니다.
WITH daily_orders AS (
  SELECT
    DATE(created_at) AS order_date,
    COUNT(order_id) AS total_order_count
  FROM `ecommerce.orders`
  GROUP BY 1
)
SELECT *
FROM AI.FORECAST(
  TABLE daily_orders,
  horizon => 7,                      -- 향후 7일 예측
  timestamp_col => 'order_date',       -- 시간 기준 컬럼
  data_col => 'total_order_count' -- 예측할 대상 컬럼
)

Query is running:   0%|          |

Downloading:   0%|          |

,forecast_timestamp,forecast_value,confidence_level,prediction_interval_lower_bound,prediction_interval_upper_bound,ai_forecast_status
0,2026-04-29 00:00:00+00:00,180.820450,0.95,60.998885,300.642015,
1,2026-04-30 00:00:00+00:00,204.261749,0.95,92.211464,316.312034,
2,2026-05-01 00:00:00+00:00,208.529068,0.95,110.088556,306.969579,
3,2026-05-02 00:00:00+00:00,210.120941,0.95,117.100006,303.141877,
4,2026-05-03 00:00:00+00:00,213.560165,0.95,121.191666,305.928664,
5,2026-05-04 00:00:00+00:00,216.681717,0.95,124.876420,308.487013,
6,2026-05-05 00:00:00+00:00,219.223419,0.95,124.493848,313.952991,


In [ ]:
# AI.PARSE_DCOUMENT 진행의 사전 조건으로 Object table 생성이 필요함. 이에 필요한 Connection 생성
!bq mk \
  --connection \
  --location=us \
  --project_id={project_id} \
  --connection_type=CLOUD_RESOURCE \
  objectconn

BigQuery error in mk operation: Already Exists: Connection
projects/639513110139/locations/us/connections/objectconn


In [ ]:
#Document AI API 활성화
!gcloud services enable documentai.googleapis.com --project={PROJECT_ID}

In [ ]:
%%bigquery

CREATE OR REPLACE EXTERNAL TABLE `ecommerce.obj_table_docs`
WITH CONNECTION `projects/krda-next26/locations/us/connections/objectconn`
OPTIONS(
object_metadata = 'SIMPLE',
uris = ['gs://krda_next26_data/*.pdf']
)

Query is running:   0%|          |

""


In [ ]:
%%bigquery
-- 해당 쿼리 동작 시킨 후 나타나는 에러 메시지에 service 계정 파악 가능(권한 부여된 상태는 나타나지 않음)
-- 생성 후 해당 모델을 사용하는 service 계정에 Storage Object Viewer, Document AI Administrator 자격 부여할 것
CREATE OR REPLACE MODEL `ecommerce.docai_model`
REMOTE WITH CONNECTION DEFAULT
OPTIONS (
 REMOTE_SERVICE_TYPE = 'CLOUD_AI_DOCUMENT_V1',
 DOCUMENT_PROCESSOR = 'c9b94d50d20d9bbb'
);


Query is running:   0%|          |

""


In [ ]:
%%bigquery
-- layout parser endpoint (Document AI에서 생성 필요)
-- 260517 현 시점에서는 ML.PROCESS_DOCUMENT 함수 사용을 더 권장
SELECT *
FROM AI.PARSE_DOCUMENT(
TABLE  ecommerce.obj_table_docs, -- 파싱할 Object table
       endpoint => 'projects/639513110139/locations/us/processors/c9b94d50d20d9bbb',
       connection_id => "projects/krda-next26/locations/us/connections/__default_cloudresource_connection__",
       chunk_size => 500  -- 기본값은 1000
)


Query is running:   0%|          |

Downloading:   0%|          |

,chunk_id,start_page,end_page,content,status,uri,generation,content_type,size,md5_hash,updated,metadata,ref
0,1,1,1,# The Look e-commerce 이용약관 및,,gs://krda_next26_data/[가상문서] The Look e-commer...,1778990029178906,application/pdf,111373,f99a938a73e49659ec26038577a88128,2026-05-17 03:53:49.184000+00:00,[],{'uri': 'gs://krda_next26_data/[가상문서] The Look...
1,2,1,1,# 취소/반품/교환 정책\n\n최종 수정일: 2024년 5월 22일,,gs://krda_next26_data/[가상문서] The Look e-commer...,1778990029178906,application/pdf,111373,f99a938a73e49659ec26038577a88128,2026-05-17 03:53:49.184000+00:00,[],{'uri': 'gs://krda_next26_data/[가상문서] The Look...
2,3,1,3,"## 제1조 (목적)\n\n본 정책은 The Look e-commerce(이하 ""회...",,gs://krda_next26_data/[가상문서] The Look e-commer...,1778990029178906,application/pdf,111373,f99a938a73e49659ec26038577a88128,2026-05-17 03:53:49.184000+00:00,[],{'uri': 'gs://krda_next26_data/[가상문서] The Look...
3,4,3,4,구매 적립: 상품 구매 확정 시 결제 실 결제 금액의 1%가 기본 적립됩니다. (멤...,,gs://krda_next26_data/[가상문서] The Look e-commer...,1778990029178906,application/pdf,111373,f99a938a73e49659ec26038577a88128,2026-05-17 03:53:49.184000+00:00,[],{'uri': 'gs://krda_next26_data/[가상문서] The Look...


In [ ]:
%%bigquery
-- AI.COUNT_TOKENS 함수는 Vertex AI API 비용이 청구되지 않습니다.
-- BigQuery 내부에서 자체적으로 토큰 계산하기 때문입니다.
SELECT
    review,
    -- 전체 토큰 수 결과값만 추출 (.result)
    AI.COUNT_TOKENS(review, endpoint => 'gemini-2.5-pro').result AS token_count,

    -- 세부적인 모달리티 정보가 담긴 JSON 전체를 볼 때 (.full_response)
    AI.COUNT_TOKENS(review, endpoint => 'gemini-2.5-pro').full_response AS response_details
FROM
    `bigquery-public-data.imdb.reviews` -- 구글 공개데이터 세트에서 리뷰 테이블을 활용
WHERE
    review IS NOT NULL
LIMIT 3;



Query is running:   0%|          |

Downloading:   0%|          |

,review,token_count,response_details
0,Ik know it is impossible to keep all details o...,324,"{""promptTokensDetails"":[{""modality"":""TEXT"",""to..."
1,"""The Racketeer"" stars Carol (deprived of the ""...",365,"{""promptTokensDetails"":[{""modality"":""TEXT"",""to..."
2,"This one and the one prior ""Toulon's Revenge"" ...",230,"{""promptTokensDetails"":[{""modality"":""TEXT"",""to..."


In [ ]:
%%bigquery
-- AI.DETECT_ANOMALIES 는 이상치를 탐지하는 함수
-- 재고 관리를 위한 특정 카테고리의 비정상적인 반품 건수 탐지
WITH DailyReturns AS (
  SELECT
    DATE(o.returned_at) AS return_date,
    p.category,
    COUNT(o.order_id) AS daily_return_count
  FROM
    `ecommerce.order_items` AS o
  JOIN
    `ecommerce.products` AS p
  ON o.product_id = p.id
  WHERE
    o.status = 'Returned'
    AND p.category IN ('Jeans', 'Sweaters') -- 두 카테고리만 대상
    AND o.returned_at IS NOT NULL
  GROUP BY
    return_date, category
),

-- 2023년 과거 데이터 분리 (History용)
PastReturns AS (
  SELECT * FROM DailyReturns
  WHERE return_date BETWEEN '2023-01-01' AND '2023-12-31'
),

-- 이상치인지 검증하고 싶은 최근 데이터 영역 (Target용)
RecentReturns AS (
  SELECT * FROM DailyReturns
  WHERE return_date BETWEEN '2025-01-01' AND '2026-05-31'
)

-- 이상 탐지 수행
SELECT
  category,
  time_series_timestamp AS return_date,
  time_series_data AS daily_return_count,
  is_anomaly,
  lower_bound,
  upper_bound
FROM AI.DETECT_ANOMALIES(
  (SELECT * FROM PastReturns),
  (SELECT * FROM RecentReturns),
  data_col => 'daily_return_count', -- 분석할 값으로 여기서는 반품 건수
  timestamp_col => 'return_date',   -- 시간축
  id_cols => ['category'],          -- 여러 카테고리를 독립적인 시계열로 분리해서 분석
  anomaly_prob_threshold => 0.95
)
-- 이상치로 판별된 날짜만 필터링하여 확인
WHERE is_anomaly = TRUE
ORDER BY category, return_date;


Query is running:   0%|          |

Downloading:   0%|          |

,category,return_date,daily_return_count,is_anomaly,lower_bound,upper_bound
0,Jeans,2025-01-24 00:00:00+00:00,3.0,True,0.953008,2.144946
1,Jeans,2025-03-23 00:00:00+00:00,3.0,True,0.962509,2.130873
2,Jeans,2025-04-12 00:00:00+00:00,3.0,True,0.954933,2.137611
3,Jeans,2025-04-20 00:00:00+00:00,3.0,True,0.957887,2.126085
4,Jeans,2025-05-22 00:00:00+00:00,3.0,True,0.964188,2.127532
...,...,...,...,...,...,...
71,Sweaters,2026-04-17 00:00:00+00:00,5.0,True,0.873876,2.269731
72,Sweaters,2026-04-21 00:00:00+00:00,4.0,True,0.874687,2.271390
73,Sweaters,2026-04-29 00:00:00+00:00,5.0,True,0.874731,2.272626
74,Sweaters,2026-05-01 00:00:00+00:00,7.0,True,0.875833,2.274319


#### **Zero-Shot Tabular FM**

\# BQML \# SQL \# 효율화

What is Zero-Shot Tabular FM?

*   Tabbular FM 활용한 제로샷 회귀/분류를 수행합니다.

What is benefits?

*   복잡한 ML 파이프라인 단순화가 가능합니다.
*   SQL문으로 손쉬운 사용이 가능합니다.

May. 19. 2026 업데이트 예정

In [ ]:
%%bigquery

-- 1. 학습 데이터(과거) 테이블 생성
CREATE OR REPLACE TABLE `ecommerce2.past_orders` AS
SELECT
    sale_price,
    product_id,
    created_at,
    shipped_at,
    delivered_at,
    returned_at,
    status
FROM `ecommerce2.order_items`
WHERE created_at < '2023-01-01'
LIMIT 1000;          -- TabularFM은 소량의 데이터(Few-shot)만으로도 작동합니다.

-- 2. 예측 대상 데이터 테이블 생성
CREATE OR REPLACE TABLE `ecommerce2.new_orders` AS
SELECT
    order_id,
    sale_price,
    product_id,
    created_at,
    shipped_at,
    delivered_at,
    returned_at
FROM `ecommerce2.order_items`
WHERE created_at >= '2023-01-01'
LIMIT 10;

-- 3. AI.PREDICT를 통해 tabularFM 호출하여 즉시 예측 수행
SELECT *
FROM AI.PREDICT(
  training_data => (TABLE `ecommerce2.past_orders`),
  prediction_data => (TABLE `ecommerce2.new_orders`),
  model_name => 'tabularFM',
  label_col => 'status'
);


Executing query with job ID: 4931921a-2c31-44b9-bab9-12a5a8199275
Query executing: 0.27s


ERROR:
 400 GET https://bigquery.googleapis.com/bigquery/v2/projects/krda-next26/queries/4931921a-2c31-44b9-bab9-12a5a8199275?maxResults=0&location=us-central1&prettyPrint=false: Query error: Table-valued function not found: AI.PREDICT at [31:6]

Location: us-central1
Job ID: 4931921a-2c31-44b9-bab9-12a5a8199275



#### **ObjectRef**

\# 데이터타입 \# 빅쿼리 \# 멀티모달분석

What is ObjectRef?

*   멀티모달 분석을 위한 데이터 타입으로 멀티모달에 대한 메타데이터를 가지고 있음
*   기존의 Object Table 생성해야 가능했던 멀티모달 분석을 넘어서 기존 테이블에 연관된 멀티모달을 컬럼으로 붙여 사용할 수 있는 강력한 데이터 타입

What is benefits?

*   기존 Object Table을 생성하지 않고 기존 테이블에서도 하나의 컬럼으로 추가하여 멀티모달(이미지, 음성 등)을 다룰 수 있게 함

In [ ]:
%%bigquery
-- ObjectRef의 OBJ.MAKE_REF를 통해 직접 접근하는 쿼리
-- 해당 구문을 응용하면 기존 테이블에 멀티모달 데이터를 붙일 수 있음
SELECT AI.GENERATE(
  ("이미지에 대해서 설명해주세요.:",
  OBJ.GET_ACCESS_URL(OBJ.MAKE_REF("gs://krda_next26_data/img/dimaliss-street-cafe-4472312.jpg"), 'r')));

Query is running:   0%|          |

Downloading:   0%|          |

,f0_
0,"{'result': '이 이미지는 활기 넘치는 도심 거리에 위치한 ""GLONOJAD..."


In [ ]:
%%bigquery
-- Object Table 사용해 Ref 값 생성하기
-- Object Table은 ref 컬럼으로 ObjectRef을 가능하게 제공하고 있음
CREATE OR REPLACE EXTERNAL TABLE `objectrefimg.images`
WITH CONNECTION `projects/krda-next26/locations/us/connections/objectconn`
OPTIONS(
object_metadata = 'SIMPLE',
uris = ['gs://krda_next26_data/img/*']
)

Query is running:   0%|          |

""


In [ ]:
%%bigquery
-- Object Table에 Ref 값 검증
SELECT uri, ref AS image_ref
FROM objectrefimg.images;

Query is running:   0%|          |

Downloading:   0%|          |

,uri,image_ref
0,gs://krda_next26_data/img/dimaliss-street-cafe...,{'uri': 'gs://krda_next26_data/img/dimaliss-st...
1,gs://krda_next26_data/img/matthiaskost-squirre...,{'uri': 'gs://krda_next26_data/img/matthiaskos...
2,gs://krda_next26_data/img/u_doid11kew1-woman-6...,{'uri': 'gs://krda_next26_data/img/u_doid11kew...
3,gs://krda_next26_data/img/wikiimages-astronaut...,{'uri': 'gs://krda_next26_data/img/wikiimages-...
4,gs://krda_next26_data/img/wikiimages-rocket-la...,{'uri': 'gs://krda_next26_data/img/wikiimages-...
5,gs://krda_next26_data/img/wolfgang_hasselmann-...,{'uri': 'gs://krda_next26_data/img/wolfgang_ha...


In [ ]:
%%bigquery
-- Object Table에 있는 모든 이미지를 대상으로 분석 수행
SELECT
    uri,
    AI.GENERATE(
        ("이미지에 대해서 한 줄로 요약해서 설명해주세요.:",
         OBJ.GET_ACCESS_URL(ref, 'r'))
    ) AS image_summary
FROM `objectrefimg.images`;


Query is running:   0%|          |

Downloading:   0%|          |

,uri,image_summary
0,gs://krda_next26_data/img/wikiimages-rocket-la...,{'result': '거대한 불꽃과 연기를 뿜어내며 힘차게 발사되는 우주왕복선의 극...
1,gs://krda_next26_data/img/wikiimages-astronaut...,"{'result': '지구를 배경으로 우주 유영 중인 우주비행사의 모습이며, 헬멧 ..."
2,gs://krda_next26_data/img/dimaliss-street-cafe...,{'result': '이 이미지는 'GLONOJAD WEGETARIAŃSKI BAR...
3,gs://krda_next26_data/img/u_doid11kew1-woman-6...,{'result': '밝게 웃는 여성이 커피를 들고 카페 문을 열며 서 있는 모습입...
4,gs://krda_next26_data/img/wolfgang_hasselmann-...,{'result': '하늘을 배경으로 주름진 피부와 튼튼한 상아가 돋보이는 코끼리의...
5,gs://krda_next26_data/img/matthiaskost-squirre...,{'result': '앙상한 나뭇가지 위에 앉아 카메라를 똑바로 응시하는 털 복슬복...


#### **Python UDFs**

\# Python \# 함수 \# 분석

What is Python UDFs?

*   SQL만으로 처리하기 복잡하거나 불가능한 로직을 Python 코드로 작성해 마치 BigQuery의 내장 SQL 함수처럼 사용할 수 있게 하는 기능
*   BigQuery의 강력한 분산 처리 엔진 위에서 Python 코드가 자동으로 Scale-out하며 실행되며 BigQuery의 Serverless 이점을 제공하는 기능

What is benefits?

*   BigQuery에서 SQL로만 분석하기엔 복잡한 쿼리가 발생할 수 있습니다. 이때 Python의 라이브러리를 활용하기 때문에 풍부한 Python 생태계 활용이 가능합니다. 또한, 기존 Python의 편의성을 활용해 직관적으로 코드 해결을 할 수 있습니다.
*   BigQuery의 SQL 데이터 타입과 Python의 데이터 타입 간의 변환을 BigQuery가 자동으로 처리해 편의성을 가져갈 수 있습니다.

##### 시나리오  
*   순수 영업일 기준 기준 배송 소요 시간 계산
*   물류팀에서 배송 수행 능력을 평가하려고 하는 상황에서 주말을 제외한 순수 영업일로만 계산해야하는 상황을 가정합니다.

#### 문제점
*   BigQuery에서 SQL만으로 주말을 제외한 일수를 계산하려면 GENERATE_DATE_ARRAY를 쓴 후 요일을 EXTRACT(DAYOFWEEK...)한 뒤 다시 집계하는 식으로 쿼리가 상당히 복잡해짐

#### 해결법
*   Python의 datatime 라이브러리를 활용해 직관적 해결 가능

In [ ]:
%%bigquery

-- 영업일 계산 Python UDF 생성
CREATE OR REPLACE FUNCTION `krda-next26.ecommerce.calc_business_days`(start_date DATE, end_date DATE)
RETURNS INT64
LANGUAGE python
OPTIONS(
    runtime_version = 'python-3.11',
    entry_point = "get_business_days"
)
AS r"""
from datetime import timedelta

def get_business_days(start_date, end_date):

    # 입력값이 없거나 비정상적인 경우 예외 처리
    if start_date is None or end_date is None or end_date < start_date:
        return None

    business_days = 0
    current_date = start_date

    # 시작일부터 종료일까지 하루씩 더해가며 평일(월~금)만 카운트
    while current_date <= end_date:
        if current_date.weekday() < 5:  # 0:월, 1:화, 2:수, 3:목, 4:금 (5:토, 6:일)
            business_days += 1
        current_date += timedelta(days=1)

    return business_days
""";


Query is running:   0%|          |

""


In [ ]:
%%bigquery df_delivery_stats
-- 배송 데이터에 UDF 적용
SELECT
    order_id,
    user_id,
    DATE(shipped_at) AS shipped_date,
    DATE(delivered_at) AS delivered_date,
    -- 기존 SQL 방식의 단순 일수 차이 (주말 포함)
    DATE_DIFF(DATE(delivered_at), DATE(shipped_at), DAY) AS total_days,
    -- Python UDF를 활용한 순수 영업일 계산
    `krda-next26.ecommerce.calc_business_days`(DATE(shipped_at), DATE(delivered_at)) AS business_days
FROM
    `ecommerce.orders`
WHERE
    status = 'Complete'
    AND shipped_at IS NOT NULL
    AND delivered_at IS NOT NULL
ORDER BY
    shipped_at DESC
LIMIT 10;


Query is running:   0%|          |

Downloading:   0%|          |

In [ ]:
# 결과 출력
display(df_delivery_stats)

,order_id,user_id,shipped_date,delivered_date,total_days,business_days
0,73771,59230,2026-04-30,2026-05-04,4,3
1,113529,91025,2026-04-30,2026-05-05,5,4
2,31836,25501,2026-04-30,2026-05-03,3,2
3,21157,16992,2026-04-30,2026-05-03,3,2
4,46438,37307,2026-04-30,2026-05-05,5,4
5,90985,73025,2026-04-30,2026-05-02,2,2
6,22615,18174,2026-04-30,2026-05-04,4,3
7,20194,16210,2026-04-30,2026-05-05,5,4
8,5603,4463,2026-04-30,2026-05-05,5,4
9,120543,96708,2026-04-30,2026-05-04,4,3


#### **Knowledge Graph**

\# 맥락분석 \# 그래프 \# 에이전트

What is Knowledge Graph?

*   노드와 엣지 그리고 속성을 구성해 데이터의 관계를 표현하는 모델링 방식으로 기존 SQL으로 분석이 어려웠던 부분을 GQL로 분석할 수 있는 기능
*   예를 들어, 한 레스토랑의 매출이 감소했고 어떤 시간대에 손님이 많이 떨어졌는지는 파악하지만 그 원인을 모름. 비가 내렸을 수도 있고 일하던 직원의 업무 수준이나 손님의 유형에 따라 회전율이 달랐을 수도 있는데 이에 대한 데이터를 그래프로 관계성을 맺어 분석할 수 있음


What is benefits?

*   기존 SQL만으로는 분석하기 어려운 관계를 통한 '맥락'이 필요한 상황을 손쉽게 분석할 수 있게되어 단순히 수치를 넘어선 논리적인 관계까지 파악 가능
*   Agent에 데이터를 연결하고 분석하면 분석을 잘하는 것 같지만 맥락적인 부분은 누락되는 경우가 많은데 이를 가능하게 함

In [ ]:
%%bigquery
-- 비정형데이터를 그래프로 생성한 후 분석하는 시나리오
CREATE OR REPLACE EXTERNAL TABLE `pdftograph.document_object_table`
WITH CONNECTION `projects/krda-next26/locations/us/connections/objectconn`
OPTIONS(
object_metadata = 'SIMPLE',
uris = ['gs://krda_next26_data/*.pdf']
)

Query is running:   0%|          |

""


In [ ]:
%%bigquery
CREATE OR REPLACE TABLE pdftograph.parsing_result AS(
SELECT *
FROM AI.PARSE_DOCUMENT(
TABLE  pdftograph.document_object_table, -- 파싱할 Object table
       endpoint => 'projects/639513110139/locations/us/processors/c9b94d50d20d9bbb',
       connection_id => "projects/krda-next26/locations/us/connections/__default_cloudresource_connection__",
       chunk_size => 500
))

Query is running:   0%|          |

""


In [ ]:
%%bigquery
-- 사전에 Ontology의 Semantic Layer가 구성되어 있다고 가정
-- 엣지(Edge) 테이블 생성 쿼리
CREATE OR REPLACE TABLE `pdftograph.extracted_edges` AS
SELECT
  chunk_id,
  e.source,
  e.source_type,
  e.target,
  e.target_type,
  e.relation,
  e.impact,
  e.numerical_value,
  e.context
FROM (
  SELECT
    chunk_id,
    AI.GENERATE(
      """
      ### ROLE
      You are a specialized E-commerce Knowledge Graph Architect.
      Your goal is to extract structured triples from e-commerce policy and membership documents with 100% consistency.
      CRITICAL MISSION: You MUST connect the "Membership/Benefit" context with the "Operations/Policy" context. Avoid creating disconnected islands of data.

      ### ONTOLOGY: NODES & TYPES
      Extract entities and categorize them into these exact types:
      - Policy: 규정이나 약관 (e.g., 'Return Policy', 'Exchange Policy')
      - Benefit: 고객에게 제공되는 혜택 (e.g., 'Free Shipping', 'Free Return', 'Point Accumulation')
      - Condition: 혜택이나 정책을 적용받기 위한 조건/기준 (e.g., 'Within 7 Days', 'Unused', 'Over 500,000 KRW')
      - Membership_Tier: 멤버십 등급 (e.g., 'BRONZE_TIER', 'SILVER_TIER', 'GOLD_TIER', 'PLATINUM_TIER')
      - Category: 상품 카테고리 (e.g., 'Clothing', 'Shoes')
      - Entity: 회사, 포인트 등 주요 개체 (e.g., 'The Look', 'The Look Point')
      - Action: 사용자가 취하는 행동 (e.g., 'Return', 'Exchange', 'Text Review', 'Photo Review')

      ### ONTOLOGY: RELATIONSHIPS
      Use ONLY these relationship types:
      - REQUIRES_CONDITION: 혜택이나 정책이 특정 조건을 요구할 때
      - PROVIDES_BENEFIT: 멤버십 등급이나 행동이 혜택을 제공할 때
      - APPLIES_TO: 정책이나 조건이 특정 카테고리나 개체에 적용될 때
      - RESTRICTS: 정책이 특정 행동이나 카테고리를 제한할 때
      - EARNS: 특정 행동(리뷰 등)이나 등급이 포인트/보상을 적립할 때
      - EXCEPTS_FROM: **(신규) 혜택이나 멤버십이 특정 정책의 제한으로부터 예외를 제공할 때 (Cross-link용)**

      ### STRICT NORMALIZATION RULES (CRITICAL FOR GRAPH CONNECTIVITY)
      1. Normalize to Canonical Names: Use English ONLY for source/target (e.g., 'GOLD_TIER', 'Return', 'Exchange').
      2. No Loose Strings: Ensure 'source' and 'target' are concise entity names (max 2-3 words).
      3. Unify Common Entities:
         - If the text mentions "반품", "취소", "교환" in ANY context (policy or membership), ALWAYS map it to the exact `Action` nodes: 'Return', 'Cancel', 'Exchange'.
         - If a membership provides a "무료 반품" or "무료 교환" benefit, map it to the `Benefit` nodes: 'Free Return', 'Free Exchange'.
         - Crucial: Link Membership benefits directly to Operations. For example, if GOLD_TIER provides 'Free Return', and 'Return Policy' restricts 'Return', you MUST create a link showing they interact (e.g., 'Free Return' EXCEPTS_FROM 'Return Policy').
      4. Impact: Must be 'positive' (혜택/허용), 'negative' (제한/페널티), or 'neutral' (단순 규칙).

      ### LANGUAGE
      - source/target: English (Canonical forms)
      - context: Korean (해당 관계를 증명하는 원본 문서의 구체적인 설명/근거)

      ### TEXT TO PROCESS
      """ || content,

      output_schema => """
        edges ARRAY<STRUCT<
          source STRING,
          source_type STRING,
          target STRING,
          target_type STRING,
          relation STRING,
          impact STRING,
          numerical_value STRING,
          context STRING
        >>
      """,
      endpoint => 'https://aiplatform.googleapis.com/v1/projects/krda-next26/locations/global/publishers/google/models/gemini-3.1-pro-preview'
    ) AS extracted_data
  FROM `pdftograph.parsing_result`
  WHERE content IS NOT NULL
), UNNEST(extracted_data.edges) AS e;


Query is running:   0%|          |

""


In [13]:
%%bigquery
-- 사전에 Ontology의 Semantic Layer가 구성되어 있다고 가정
-- 엣지(Edge) 테이블 생성 쿼리
CREATE OR REPLACE TABLE `pdftograph.extracted_edges` AS
SELECT
  chunk_id,
  e.source,
  e.source_type,
  e.target,
  e.target_type,
  e.relation,
  e.impact,
  e.numerical_value,
  e.context
FROM (
  SELECT
    chunk_id,
    AI.GENERATE(
      """
      ### ROLE
      You are a specialized E-commerce Knowledge Graph Architect.
      Your goal is to extract structured triples from e-commerce policy and membership documents with 100% consistency.
      CRITICAL MISSION: You MUST connect the "Membership/Benefit" context with the "Operations/Policy" context. Avoid creating disconnected islands of data.

      ### ONTOLOGY: NODES & TYPES
      Extract entities and categorize them into these exact types:
      - Policy: 규정이나 약관 (e.g., 'Return Policy', 'Exchange Policy')
      - Benefit: 고객에게 제공되는 혜택 (e.g., 'Free Shipping', 'Free Return', 'Point Accumulation')
      - Condition: 혜택이나 정책을 적용받기 위한 조건/기준 (e.g., 'Within 7 Days', 'Unused', 'Over 500,000 KRW')
      - Membership_Tier: 멤버십 등급 (e.g., 'BRONZE_TIER', 'SILVER_TIER', 'GOLD_TIER', 'PLATINUM_TIER')
      - Category: 상품 카테고리 (e.g., 'Clothing', 'Shoes')
      - Entity: 회사, 포인트 등 주요 개체 (e.g., 'The Look', 'The Look Point')
      - Action: 사용자가 취하는 행동 (e.g., 'Return', 'Exchange', 'Text Review', 'Photo Review')

      ### ONTOLOGY: RELATIONSHIPS
      Use ONLY these relationship types:
      - REQUIRES_CONDITION: 혜택이나 정책이 특정 조건을 요구할 때
      - PROVIDES_BENEFIT: 멤버십 등급이나 행동이 혜택을 제공할 때
      - APPLIES_TO: 정책이나 조건이 특정 카테고리나 개체에 적용될 때
      - RESTRICTS: 정책이 특정 행동이나 카테고리를 제한할 때
      - EARNS: 특정 행동(리뷰 등)이나 등급이 포인트/보상을 적립할 때
      - EXCEPTS_FROM: **(신규) 혜택이나 멤버십이 특정 정책의 제한으로부터 예외를 제공할 때 (Cross-link용)**

      ### STRICT NORMALIZATION RULES (CRITICAL FOR GRAPH CONNECTIVITY)
      1. Normalize to Canonical Names: Use English ONLY for source/target (e.g., 'GOLD_TIER', 'Return', 'Exchange').
      2. No Loose Strings: Ensure 'source' and 'target' are concise entity names (max 2-3 words).
      3. Unify Common Entities:
         - If the text mentions "반품", "취소", "교환" in ANY context (policy or membership), ALWAYS map it to the exact `Action` nodes: 'Return', 'Cancel', 'Exchange'.
         - If a membership provides a "무료 반품" or "무료 교환" benefit, map it to the `Benefit` nodes: 'Free Return', 'Free Exchange'.
         - Crucial: Link Membership benefits directly to Operations. For example, if GOLD_TIER provides 'Free Return', and 'Return Policy' restricts 'Return', you MUST create a link showing they interact (e.g., 'Free Return' EXCEPTS_FROM 'Return Policy').
      4. Impact: Must be 'positive' (혜택/허용), 'negative' (제한/페널티), or 'neutral' (단순 규칙).

      ### LANGUAGE
      - source/target: English (Canonical forms)
      - context: Korean (해당 관계를 증명하는 원본 문서의 구체적인 설명/근거)

      ### TEXT TO PROCESS
      """ || content,
      output_schema => """
        edges ARRAY<STRUCT<
          source STRING,
          source_type STRING,
          target STRING,
          target_type STRING,
          relation STRING,
          impact STRING,
          numerical_value STRING,
          context STRING
        >>
      """,
      -- 최신 모델 버전으로 조정 (3.1-pro-preview가 사용 불가능할 경우 2.5-pro 추천)
      endpoint => 'https://aiplatform.googleapis.com/v1/projects/krda-next26/locations/global/publishers/google/models/gemini-3.1-pro-preview'
    ) AS extracted_data
  FROM `pdftograph.parsing_result`
  WHERE content IS NOT NULL
), UNNEST(extracted_data.edges) AS e;


Query is running:   0%|          |

""


In [14]:
%%bigquery
-- 엣지(Edge) 검증
SELECT *
FROM `pdftograph.extracted_edges`
LIMIT 3

Query is running:   0%|          |

Downloading:   0%|          |

,chunk_id,source,source_type,target,target_type,relation,impact,numerical_value,context
0,3,Return Policy,Policy,Unused,Condition,REQUIRES_CONDITION,negative,null,"고객의 사용 흔적(착용, 오염, 세탁 등)에 의하여 상품의 가치가 현저히 감소한 경..."
1,2,Exchange Policy,Policy,Exchange,Action,APPLIES_TO,neutral,null,취소/반품/교환 정책 제목
2,4,Free Return,Benefit,Return Policy,Policy,EXCEPTS_FROM,positive,,멤버십 혜택으로 제공되는 무료 반품은 반품 정책의 기본 배송비 부과 제한으로부터 예...


In [ ]:
%%bigquery

-- 노드(Node) 테이블 생성 쿼리
CREATE OR REPLACE TABLE `pdftograph.extracted_nodes` AS
SELECT DISTINCT
  node_id,
  node_type
FROM (
  SELECT source AS node_id, source_type AS node_type FROM `pdftograph.extracted_edges`
  UNION DISTINCT
  SELECT target AS node_id, target_type AS node_type FROM `pdftograph.extracted_edges`
)
-- 새로운 노드 타입들로 필터링
WHERE node_type IN (
  'Policy', 'Benefit', 'Condition', 'Membership_Tier',
  'Category', 'Entity', 'Action'
);

Query is running:   0%|          |

""


In [ ]:
%%bigquery

SELECT *
FROM `pdftograph.extracted_nodes`
LIMIT 3

Query is running:   0%|          |

Downloading:   0%|          |

,node_id,node_type
0,Return,Action
1,Defective Exchange,Action
2,Text Review,Action


In [16]:
%%bigquery

-- BigQuery Graph는 물리적인 테이블 또는 뷰의 이름만 받게 설계 됨 ----
-- 즉, 인라인 서브쿼리는 안된다는 것 ------------------------------
-------------------------------------------------------------
-- 1. 개별 노드 타입별 뷰 생성
-------------------------------------------------------------
CREATE OR REPLACE VIEW `pdftograph.PolicyNode` AS SELECT node_id FROM `pdftograph.extracted_nodes` WHERE node_type = 'Policy';
CREATE OR REPLACE VIEW `pdftograph.BenefitNode` AS SELECT node_id FROM `pdftograph.extracted_nodes` WHERE node_type = 'Benefit';
CREATE OR REPLACE VIEW `pdftograph.ConditionNode` AS SELECT node_id FROM `pdftograph.extracted_nodes` WHERE node_type = 'Condition';
CREATE OR REPLACE VIEW `pdftograph.MembershipNode` AS SELECT node_id FROM `pdftograph.extracted_nodes` WHERE node_type = 'Membership_Tier';
CREATE OR REPLACE VIEW `pdftograph.CategoryNode` AS SELECT node_id FROM `pdftograph.extracted_nodes` WHERE node_type = 'Category';
CREATE OR REPLACE VIEW `pdftograph.ActionNode` AS SELECT node_id FROM `pdftograph.extracted_nodes` WHERE node_type = 'Action';
CREATE OR REPLACE VIEW `pdftograph.EntityNode` AS SELECT node_id FROM `pdftograph.extracted_nodes` WHERE node_type = 'Entity';

-------------------------------------------------------------
-- 2. 엣지 뷰 생성
-------------------------------------------------------------
-- 정책(Policy) -> 조건(Condition) 요구
CREATE OR REPLACE VIEW `pdftograph.Edge_Policy_Req_Condition` AS
SELECT source, target, impact, numerical_value, context FROM `pdftograph.extracted_edges`
WHERE relation = 'REQUIRES_CONDITION' AND source_type = 'Policy' AND target_type = 'Condition';

-- 멤버십(Membership_Tier) -> 혜택(Benefit) 제공
CREATE OR REPLACE VIEW `pdftograph.Edge_Tier_Prov_Benefit` AS
SELECT source, target, impact, numerical_value, context FROM `pdftograph.extracted_edges`
WHERE relation = 'PROVIDES_BENEFIT' AND source_type = 'Membership_Tier' AND target_type = 'Benefit';

-- 정책(Policy) -> 카테고리(Category)에 적용
CREATE OR REPLACE VIEW `pdftograph.Edge_Policy_Applies_Cat` AS
SELECT source, target, impact, numerical_value, context FROM `pdftograph.extracted_edges`
WHERE relation = 'APPLIES_TO' AND source_type = 'Policy' AND target_type = 'Category';

-- 사용자 행동(Action) -> 포인트/개체(Entity) 적립
CREATE OR REPLACE VIEW `pdftograph.Edge_Action_Earns_Entity` AS
SELECT source, target, impact, numerical_value, context FROM `pdftograph.extracted_edges`
WHERE relation = 'EARNS' AND source_type = 'Action' AND target_type = 'Entity';

-- 정책(Policy) -> 제한(Action) 적용
CREATE OR REPLACE VIEW `pdftograph.Edge_Policy_Restricts_Act` AS
SELECT source, target, impact, numerical_value, context FROM `pdftograph.extracted_edges`
WHERE relation = 'RESTRICTS' AND source_type = 'Policy' AND target_type = 'Action';

-- 혜택(Benefit) -> 정책(Policy) 제한에 대한 예외 제공
CREATE OR REPLACE VIEW `pdftograph.Edge_Benefit_Excepts_Policy` AS
SELECT source, target, impact, numerical_value, context FROM `pdftograph.extracted_edges`
WHERE relation = 'EXCEPTS_FROM' AND source_type = 'Benefit' AND target_type = 'Policy';


-------------------------------------------------------------
-- 3. 스키마가 엄격히 적용된 Property Graph 생성
-------------------------------------------------------------
CREATE OR REPLACE PROPERTY GRAPH pdftograph.ecommerce_policy_graph

  NODE TABLES (
    `pdftograph.PolicyNode` AS PolicyNode KEY (node_id) LABEL Policy PROPERTIES (node_id),
    `pdftograph.BenefitNode` AS BenefitNode KEY (node_id) LABEL Benefit PROPERTIES (node_id),
    `pdftograph.ConditionNode` AS ConditionNode KEY (node_id) LABEL Condition PROPERTIES (node_id),
    `pdftograph.MembershipNode` AS MembershipNode KEY (node_id) LABEL Membership_Tier PROPERTIES (node_id),
    `pdftograph.CategoryNode` AS CategoryNode KEY (node_id) LABEL Category PROPERTIES (node_id),
    `pdftograph.ActionNode` AS ActionNode KEY (node_id) LABEL Action PROPERTIES (node_id),
    `pdftograph.EntityNode` AS EntityNode KEY (node_id) LABEL Entity PROPERTIES (node_id)
  )

  EDGE TABLES (
    `pdftograph.Edge_Policy_Req_Condition` AS RequiresConditionEdge
      KEY (source, target)
      SOURCE KEY (source) REFERENCES PolicyNode (node_id)
      DESTINATION KEY (target) REFERENCES ConditionNode (node_id)
      LABEL REQUIRES_CONDITION
      PROPERTIES (impact, numerical_value, context),

    `pdftograph.Edge_Tier_Prov_Benefit` AS ProvidesBenefitEdge
      KEY (source, target)
      SOURCE KEY (source) REFERENCES MembershipNode (node_id)
      DESTINATION KEY (target) REFERENCES BenefitNode (node_id)
      LABEL PROVIDES_BENEFIT
      PROPERTIES (impact, numerical_value, context),

    `pdftograph.Edge_Policy_Applies_Cat` AS AppliesToEdge
      KEY (source, target)
      SOURCE KEY (source) REFERENCES PolicyNode (node_id)
      DESTINATION KEY (target) REFERENCES CategoryNode (node_id)
      LABEL APPLIES_TO
      PROPERTIES (impact, numerical_value, context),

    `pdftograph.Edge_Action_Earns_Entity` AS EarnsEdge
      KEY (source, target)
      SOURCE KEY (source) REFERENCES ActionNode (node_id)
      DESTINATION KEY (target) REFERENCES EntityNode (node_id)
      LABEL EARNS
      PROPERTIES (impact, numerical_value, context),

    `pdftograph.Edge_Policy_Restricts_Act` AS RestrictsEdge
      KEY (source, target)
      SOURCE KEY (source) REFERENCES PolicyNode (node_id)
      DESTINATION KEY (target) REFERENCES ActionNode (node_id)
      LABEL RESTRICTS
      PROPERTIES (impact, numerical_value, context),

    `pdftograph.Edge_Benefit_Excepts_Policy` AS ExceptsFromEdge
      KEY (source, target)
      SOURCE KEY (source) REFERENCES BenefitNode (node_id)
      DESTINATION KEY (target) REFERENCES PolicyNode (node_id)
      LABEL EXCEPTS_FROM
      PROPERTIES (impact, numerical_value, context)
  );


Query is running:   0%|          |

""


In [17]:
%%bigquery --graph display_only
-- 시각화 엔진은 매칭된 모든 노드와 엣지를 TO_JSON()으로 감싸서 RETURN 해야함.
GRAPH pdftograph.ecommerce_policy_graph
MATCH (n1)-[e1]->(n2)
OPTIONAL MATCH (n2)-[e2]->(n3)
RETURN
  TO_JSON(n1) AS source_node,
  TO_JSON(e1) AS first_relation,
  TO_JSON(n2) AS intermediate_node,
  TO_JSON(e2) AS second_relation,
  TO_JSON(n3) AS target_node

Query is running:   0%|          |

Downloading:   0%|          |

In [18]:
%%bigquery

-- Graph 기초 쿼리
-- GOLD 티어 고객들은 어떤 혜택이 있나요?
GRAPH pdftograph.ecommerce_policy_graph
MATCH (m:Membership_Tier {node_id: "GOLD_TIER"})-[r:PROVIDES_BENEFIT]-(b:Benefit)
RETURN m.node_id AS Tier, b.node_id AS Benefit, r.context AS Explanation

Query is running:   0%|          |

Downloading:   0%|          |

,Tier,Benefit,Explanation
0,GOLD_TIER,Free Return,GOLD 등급 고객에게는 월 1회의 무료 반품 혜택이 제공됩니다.


In [19]:
%%bigquery
-- Agent에게 "제가 환불이 가능한가요?" 라는 질문을 올때 고객의 등급 확인 및 해당 쿼리를 근거로 정확한 답변 가능.
-- 정책에 따라 달라지는 환불 요건을 맥락 파악을 못하면 Agent는 엉뚱한 답변을 할 경우가 높음
-- 테이블 구성 및 복잡한 컴퓨팅 연산을 요하는 것보다 비즈니스 확장에 맞춰서 GRAPH로 유연히 맥락 제공하는 것이 좋음
-- 해당 쿼리는 다음과 같은 USECASE에 대응할 수 있는 초석입니다.
-- 1. VIP 고객 응대 가이드(예외로 환불이 가능한지?)
-- 2. 정책 충돌 시뮬레이션(새로운 정책이 기존의 혜택들과 어떻게 충돌하고 예외 발생하는지?)
SELECT
    Membership_Tier,
    Provided_Benefit,
    Bypassed_Policy,
    Policy_Context
FROM GRAPH_TABLE(
    pdftograph.ecommerce_policy_graph
    -- 쿼리 설명
    -- 멤버십(m)이 혜택(b)을 제공(PROVIDES_BENEFIT)하고
    -- 그 혜택(b)이 특정 정책(p)으로부터 예외를 근거화 함
    MATCH (m:Membership_Tier)-[:PROVIDES_BENEFIT]->(b:Benefit)-[e:EXCEPTS_FROM]->(p:Policy)
    RETURN
        m.node_id AS Membership_Tier,
        b.node_id AS Provided_Benefit,
        p.node_id AS Bypassed_Policy,
        e.context AS Policy_Context -- 근거
)
ORDER BY Membership_Tier;

Query is running:   0%|          |

Downloading:   0%|          |

,Membership_Tier,Provided_Benefit,Bypassed_Policy,Policy_Context
0,GOLD_TIER,Free Return,Return Policy,멤버십 혜택으로 제공되는 무료 반품은 반품 정책의 기본 배송비 부과 제한으로부터 예...
1,PLATINUM_TIER,Free Return,Return Policy,멤버십 혜택으로 제공되는 무료 반품은 반품 정책의 기본 배송비 부과 제한으로부터 예...


#### **Hybrid Search** -업데이트 예정

\# 정확성&유사성 \# 하이브리드검색 \# 분석고도화

What is Hybrid Search?

*   SEARCH() 함수로 정확히 매칭 되는 부분과 VECTOR_SEARCH() 함수를 결합해 결과를 도출하는 기능
*   정확히 검색해야하는 것과 유사성이 떨어져서 못찾는 부분(핀포인트)을 상호보완하는 기능

What is benefits?

*   SEARCH()로 검색하면 오타나 동의어로 인해 검색되지 않는 문제 그리고 VECOTR_SEARCH()로 검색하면 물에 젖지 않는 운동화 검색시 방수 운동화로 잘 검색하지만 특정 브랜드의 2026년형 모델로 검색하는 핀포인트 케이스에선 정확히 나오지 않는 문제가 있는데 서로의 단점을 상호보완해 정확한 결과를 도출 가능

#### **Conversational Analytics in Bigquery**

\# 대화형분석 \# 에이전트 \# 심층분석

What is Conversational Analytics Agent?

*   기존 대시보드 분석환경을 넘어서 대화형 분석을 할 수 있는 에이전트

What is benefits?

*   특정 데이터를 Grounding하여 분석 가능하며 Glossary 연계와 Prompting을 통해 최적화 및 고도화 가능
*   Looker, DataStudio, Gemini Enterprise와 Sharing 가능할 뿐만 아니라 API로 제공해 별도의 환경을 개발해 활용 가능

##### 사용법



*   BigQuery의 콘솔에서 왼쪽 사이드 바에 Agents를 클릭합니다.  
*   필요한 API 활성화를 요구할 시에 충족합니다.  
*   New Agent를 클릭하여 메뉴에 따라 생성합니다. 자세한 가이드라인은 [링크](https://docs.cloud.google.com/bigquery/docs/create-data-agents) 참조하십시오.

